In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver: una Qiskit Function di Q-CTRL Fire Opal
> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti del piano IBM Quantum&reg; Premium, del piano Flex e del piano On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Con il Fire Opal Optimization Solver puoi risolvere problemi di ottimizzazione su scala utility su hardware quantistico senza dover possedere competenze specifiche nel campo quantistico. Inserisci semplicemente la definizione del problema ad alto livello e il Solver si occupa del resto. L'intero flusso di lavoro è noise-aware e sfrutta internamente il [Performance Management di Fire Opal](/guides/q-ctrl-performance-management). Il Solver fornisce in modo costante soluzioni accurate a problemi classicamente difficili, anche alla scala completa del dispositivo sui più grandi QPU IBM&reg;.

Il Solver è flessibile e può essere utilizzato per risolvere problemi di ottimizzazione combinatoria definiti come funzioni obiettivo o come grafi arbitrari. I problemi non devono necessariamente essere mappati alla topologia del dispositivo. Sia i problemi non vincolati che quelli vincolati sono risolvibili, a condizione che i vincoli possano essere formulati come termini di penalità. Gli esempi inclusi in questa guida mostrano come risolvere un problema di ottimizzazione su scala utility, sia non vincolato che vincolato, utilizzando diversi tipi di input per il Solver. Il primo esempio riguarda un problema Max-Cut definito su un grafo 3-regolare a 156 nodi, mentre il secondo affronta un problema di Minimum Vertex Cover a 50 nodi definito da una funzione di costo.

Per ottenere l'accesso all'Optimization Solver, [contatta Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrizione della funzione
Il Solver ottimizza e automatizza completamente l'intero algoritmo, dalla soppressione degli errori a livello hardware alla mappatura efficiente del problema e all'ottimizzazione classica a ciclo chiuso. Dietro le quinte, la pipeline del Solver riduce gli errori a ogni fase, abilitando le prestazioni avanzate necessarie per scalare in modo significativo. Il flusso di lavoro sottostante è ispirato al Quantum Approximate Optimization Algorithm (QAOA), un algoritmo ibrido quantistico-classico. Per un riepilogo dettagliato dell'intero flusso di lavoro dell'Optimization Solver, consulta [il manoscritto pubblicato](https://arxiv.org/abs/2406.01743).

![Visualizzazione del flusso di lavoro dell'Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Per risolvere un problema generico con l'Optimization Solver:
1. Definisci il tuo problema come funzione obiettivo, come grafo o come catena di spin `SparsePauliOp`.
2. Collegati alla funzione tramite il Qiskit Functions Catalog.
3. Esegui il problema con il Solver e recupera i risultati.
## Input e output
### Input
| Nome    | Tipo                    | Descrizione                                                                                                                                                                                         | Obbligatorio | Predefinito | Esempio |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` o `SparsePauliOp` | Una delle rappresentazioni elencate in "Formati di problema accettati"                                                                                                                              | Sì | N/A   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Nome della classe del problema; usato solo per le definizioni di problema basate su grafo e catena di spin, limitate a "maxcut" o "spin_chain"; non richiesto per definizioni di funzione obiettivo arbitraria | No      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Nome del backend da utilizzare                                                                                                                                                                      | No  | Il backend meno occupato a cui la tua istanza ha accesso    | `"ibm_fez"`      |
| options  | `dict`                  | Opzioni di input, tra cui: (Facoltativo) `session_id`: `str`; il comportamento predefinito crea una nuova Session                                                                                   | No | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Formati di problema accettati:**
- Rappresentazione in espressione polinomiale di una funzione obiettivo. Idealmente creata in Python con un oggetto SymPy Poly esistente e formattata in stringa utilizzando [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Rappresentazione tramite grafo di un tipo di problema specifico. Il grafo deve essere creato utilizzando la libreria networkx in Python e poi convertito in stringa tramite la funzione networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Rappresentazione in catena di spin di un problema specifico. La catena di spin deve essere rappresentata come oggetto `SparsePauliOp`; consulta la [documentazione](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) per maggiori dettagli.

**Backend supportati:**
Esegui il codice seguente per visualizzare l'elenco dei backend attualmente supportati. Se il tuo dispositivo non è presente nell'elenco, [contatta Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) per aggiungere il supporto.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Opzioni:**
| Nome   | Tipo   | Descrizione  | Predefinito |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | ID di una sessione Qiskit Runtime esistente  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Un elenco di tag per i job | `[]`|

### Output
| Nome    | Tipo | Descrizione | Esempio |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Soluzione e metadati elencati in "Contenuto del dizionario dei risultati"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Contenuto del dizionario dei risultati:**
| Nome    | Tipo | Descrizione |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | Il miglior costo minimo tra tutte le iterazioni        |
| final_bitstring_distribution  | `CountsDict`              | Il dizionario dei conteggi delle bitstring associato al costo minimo        |
| solution_bitstring | `str`              | Bitstring con il costo migliore nella distribuzione finale         |
| iteration_count  | `int`              | Il numero totale di iterazioni QAOA eseguite dal Solver        |
| variables_to_bitstring_index_map  | `float`              | La mappatura dalle variabili al bit equivalente nella bitstring       |
| best_parameters  | `list[float]`            | I parametri beta e gamma ottimizzati tra tutte le iterazioni  |
| warnings  |`list[str]`              | Gli avvisi prodotti durante la compilazione o l'esecuzione del QAOA (predefinito: None)   |
## Benchmark
I [risultati di benchmarking pubblicati](https://arxiv.org/abs/2406.01743) mostrano che il Solver risolve con successo problemi con oltre 120 qubit, superando persino i risultati precedentemente pubblicati su dispositivi a quantum annealing e ad ioni intrappolati. Le seguenti metriche di benchmark forniscono un'indicazione approssimativa dell'accuratezza e della scalabilità dei tipi di problema sulla base di alcuni esempi. Le metriche effettive possono variare in base a diverse caratteristiche del problema, come il numero di termini nella funzione obiettivo (densità) e la loro località, il numero di variabili e l'ordine polinomiale.

Il "Numero di qubit" indicato non è un limite assoluto, ma rappresenta soglie approssimative entro le quali ci si può aspettare un'accuratezza della soluzione estremamente costante. Problemi di dimensioni maggiori sono stati risolti con successo e si incoraggia la sperimentazione oltre questi limiti.

La connettività arbitraria dei qubit è supportata per tutti i tipi di problema.

| Tipo di problema    | Numero di qubit | Esempio | Accuratezza | Tempo totale (s) | Utilizzo runtime (s) | Numero di iterazioni
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problemi quadratici con connettività sparsa  | 156 | Max-Cut 3-regolare | 100%     | 1764     | 293          | 16 |
| Ottimizzazione binaria di ordine superiore | 156 | Modello di spin-glass di Ising | 100%      | 1461     | 272          | 16 |
| Problemi quadratici con connettività densa | 50 | Max-Cut completamente connesso | 100%      |  1758    | 268  | 12 |
| Problema vincolato con termini di penalità | 50 | Minimum Vertex Cover pesato con densità di archi 8% | 100%      | 1074     | 215 | 10 |
## Inizia a usarlo
Prima di tutto, autenticati usando la tua [chiave API di IBM Quantum](http://quantum.cloud.ibm.com/). Poi seleziona la Qiskit Function come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nell'ambiente locale.)

In [2]:
# %pip install networkx numpy

## Esempio: ottimizzazione non vincolata
Esegui il problema del [taglio massimo](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). Il seguente esempio dimostra le capacità del Solver su un problema Max-Cut su un grafo 3-regolare non pesato a 156 nodi, ma puoi risolvere anche problemi su grafi pesati.
Oltre a `qiskit-ibm-catalog`, per eseguire questo esempio avrai bisogno anche dei seguenti pacchetti: `networkx` e `numpy`. Puoi installare questi pacchetti decommentando la cella seguente se stai eseguendo questo esempio in un notebook con il kernel IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Output della cella di codice precedente](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

Il Solver accetta una stringa come input per la definizione del problema.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Controlla lo [stato](/guides/functions#check-job-status) del workload della tua Qiskit Function o recupera i [risultati](/guides/functions#retrieve-results) come segue:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Recupera il risultato
Recupera il valore del taglio ottimale dal dizionario dei risultati.

> **Note:** La mappatura delle variabili alla bitstring potrebbe essere cambiata. Il dizionario di output contiene un sotto-dizionario `variables_to_bitstring_index_map` che aiuta a verificare l'ordinamento.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Puoi verificare l'accuratezza del risultato risolvendo il problema classicamente con solver open-source come [PuLP](https://coin-or.github.io/pulp/) se il grafo non è densamente connesso. I problemi ad alta densità potrebbero richiedere solver classici avanzati per validare la soluzione.
## Esempio: ottimizzazione vincolata
Il precedente esempio Max-Cut è un classico problema di ottimizzazione binaria quadratica non vincolata. L'Optimization Solver di Q-CTRL può essere utilizzato per vari tipi di problemi, inclusa l'ottimizzazione vincolata. Puoi risolvere tipi di problemi arbitrari inserendo la definizione del problema rappresentata come polinomio, dove i vincoli sono modellati come termini di penalità.

Il seguente esempio mostra come costruire una funzione di costo per un problema di ottimizzazione vincolata, il [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Oltre ai pacchetti `qiskit-ibm-catalog` e `qiskit`, per eseguire questo esempio avrai bisogno anche dei seguenti pacchetti: `numpy`, `networkx` e `sympy`. Puoi installare questi pacchetti decommentando la cella seguente se stai eseguendo questo esempio in un notebook con il kernel IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Definisci il problema
Definisci un problema MVC casuale generando un grafo con nodi pesati casualmente.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Output della cella di codice precedente](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Un modello di ottimizzazione standard per il MVC pesato può essere formulato come segue. Innanzitutto, è necessario aggiungere una penalità per ogni caso in cui un arco non sia connesso a un vertice nel sottoinsieme. Quindi, sia $n_i = 1$ se il vertice $i$ è nel cover (cioè nel sottoinsieme) e $n_i = 0$ altrimenti. In secondo luogo, l'obiettivo è minimizzare il numero totale di vertici nel sottoinsieme, che può essere rappresentato dalla seguente funzione:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Ora ogni arco nel grafo deve includere almeno un estremo del cover, il che può essere espresso con la disuguaglianza:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Ogni caso in cui un arco non è connesso al vertice del cover deve essere penalizzato. Questo può essere rappresentato nella funzione di costo aggiungendo una penalità della forma $P(1-n_i-n_j+n_i n_j)$ dove $P$ è una costante di penalità positiva. Quindi, un'alternativa non vincolata alla disuguaglianza vincolata per il MVC pesato è:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Esegui il problema